In [ ]:
import os, subprocess, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ GPU')
!df -h /content | tail -1
os.makedirs('/content/data', exist_ok=True)
if not os.path.exists('/content/wnn_connectome_starterpack/datasets/train.parquet'):
    !cd /content && curl -sL https://files.wundernn.io/wnn_connectome_starterpack.tar.gz | tar -xz --wildcards '*/baseline/*' '*/utils.py' '*/datasets/*.parquet' 2>/dev/null
!ls -lh /content/wnn_connectome_starterpack/datasets
!df -h /content | tail -1

GPU: Tesla T4
overlay         113G   16G   98G  14% /
total 12G
-rw------- 1 root root 12G Nov  5 16:45 train.parquet
overlay         113G   27G   86G  24% /


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, torch
os.makedirs('/content/drive/MyDrive/wnn', exist_ok=True); os.makedirs('/content/data', exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ GPU: Runtime -> Change runtime type -> T4')

Mounted at /content/drive
GPU: Tesla T4


In [14]:
%%writefile /content/drive/MyDrive/wnn/lib.py
import os, sys, time, subprocess
import numpy as np, torch, torch.nn as nn, pyarrow.parquet as pq

dev = 'cuda'
L, NF, WARM, SAT, U3 = 20000, 112, 99, 99, 0.6569
ROOT = '/content/wnn_connectome_starterpack/datasets'
CKPT_DIR = '/content/drive/MyDrive/wnn' if os.path.isdir('/content/drive/MyDrive/wnn') else '/content'
UN = np.array([0.1254, 0.2413, 0.4685, 0.6569])
VALMIX = np.array([0.2125, 0.1853, 0.2301, 0.3721])
post = np.zeros(L, bool); post[WARM:] = True

# ---------- модели ----------
class Net(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.gru = nn.GRU(NF, h, num_layers=2, batch_first=True)
        self.head = nn.Linear(h, 2)
    def forward(self, x, h=None):
        y, h = self.gru(x, h)
        return self.head(y), h

class Net2(nn.Module):                      # то же, но с произвольным числом входов
    def __init__(self, nin, h=128):
        super().__init__()
        self.gru = nn.GRU(nin, h, num_layers=2, batch_first=True)
        self.head = nn.Linear(h, 2)
    def forward(self, x, h=None):
        y, h = self.gru(x, h)
        return self.head(y), h

class Step(nn.Module):                      # обёртка для экспорта в ONNX (интерфейс как у baseline)
    def __init__(self, net): super().__init__(); self.net = net
    def forward(self, features, hidden_0, hidden_1):
        o, h = self.net(features, torch.cat([hidden_0, hidden_1], 0))
        return o, h[0:1], h[1:2]
StepH = Step

# ---------- данные ----------
def build(src, dst, n, seed, with_mask):
    pf = pq.ParquetFile(f'{ROOT}/{src}.parquet')
    meta = {'seq_ix', 'step_in_seq', 'need_prediction', 'is_scored', 't0', 't1'}
    feat = [c for c in pf.schema_arrow.names if c not in meta]
    assert len(feat) == NF
    idx = np.sort(np.random.default_rng(seed).choice(pf.num_row_groups, n, replace=False))
    X = np.lib.format.open_memmap(f'/content/data/{dst}_X.npy', 'w+', np.float32, (n, L, NF))
    Y = np.zeros((n, L, 2), np.float32); M = np.zeros((n, L), bool) if with_mask else None
    t = time.time()
    for k, i in enumerate(idx):
        tb = pf.read_row_group(int(i), columns=feat + ['t0', 't1'] + (['is_scored'] if with_mask else []))
        X[k] = np.column_stack([tb[c].to_numpy() for c in feat])
        Y[k] = np.column_stack([tb['t0'].to_numpy(), tb['t1'].to_numpy()])
        if with_mask: M[k] = tb['is_scored'].to_numpy()
        if k % 500 == 0: print(dst, k, f'{time.time() - t:.0f} с', flush=True)
    X.flush(); np.save(f'/content/data/{dst}_Y.npy', Y)
    if with_mask: np.save(f'/content/data/{dst}_M.npy', M)

def fam_of(Y, verbose=True):                # семейство = ближайшая единица к наименьшему ненулевому |t0|
    a = np.abs(np.clip(Y[..., 0], -2, 2)).astype(np.float64)
    mn = np.where(a > 0, a, 9.0).min(1)
    f = np.abs(mn[:, None] - UN[None]).argmin(1)
    if verbose: print('макс. отклонение от единицы:', np.abs(mn - UN[f]).max().round(4), '| классы:', np.bincount(f, minlength=4))
    return f

# ---------- обучение и метрика ----------
def loss_fn(p, y, mask):
    y = y.clamp(-2, 2)
    return (y.abs() * (p - y) ** 2)[mask].mean()

@torch.no_grad()
def evaluate(model, X, Y, M, bs=50, ch=2000):
    model.eval()
    S = {k: torch.zeros(2, 6, dtype=torch.float64, device=dev) for k in ('все строки', 'is_scored')}
    pm = torch.from_numpy(post).to(dev)
    for s in range(0, len(X), bs):
        x = torch.from_numpy(np.ascontiguousarray(X[s:s + bs])).to(dev)
        y = torch.from_numpy(np.ascontiguousarray(Y[s:s + bs])).to(dev).clamp(-2, 2)
        m_sc = torch.from_numpy(np.ascontiguousarray(M[s:s + bs])).to(dev)
        h, outs = None, []
        for t in range(0, L, ch):
            o, h = model(x[:, t:t + ch], h); outs.append(o)
        p = torch.cat(outs, 1).clamp(-2, 2)
        for name, mask in (('все строки', pm[None].expand_as(m_sc)), ('is_scored', m_sc & pm[None])):
            for j in (0, 1):
                yy, pp = y[..., j][mask].double(), p[..., j][mask].double(); a = yy.abs()
                S[name][j] += torch.stack([a.sum(), (a*yy).sum(), (a*pp).sum(), (a*yy*yy).sum(), (a*pp*pp).sum(), (a*yy*pp).sum()])
    out = {}
    for k, T in S.items():
        sw = T[:, 0]; my, mp = T[:, 1] / sw, T[:, 2] / sw
        cov = T[:, 5] / sw - my * mp
        den = (T[:, 3] / sw - my ** 2) * (T[:, 4] / sw - mp ** 2)
        c = torch.where(den > 1e-16, cov / den.clamp_min(1e-16).sqrt(), torch.zeros_like(cov)).clamp(-1, 1)
        out[k] = (c.mean().item(), c.cpu().numpy().round(3))
    model.train(); return out

def train(model, epochs, bs=32, ch=1000, lr=2e-3, tag='gru'):     # использует глобальные Xtr, Ytr, Xva, Yva, Mva
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    steps_total = epochs * (len(Xtr) // bs) * (L // ch)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps_total, pct_start=0.05)
    rng = np.random.default_rng(0); best = -1
    for ep in range(epochs):
        t0 = time.time(); order = rng.permutation(len(Xtr)); run = []
        for g in range(len(Xtr) // bs):
            idx = np.sort(order[g * bs:(g + 1) * bs])
            x = torch.from_numpy(np.ascontiguousarray(Xtr[idx])).to(dev); y = torch.from_numpy(Ytr[idx]).to(dev)
            h = None
            for t in range(0, L, ch):
                o, h = model(x[:, t:t + ch], h); h = h.detach()
                mask = (torch.arange(t, t + ch, device=dev) >= WARM)[None].expand(len(idx), -1)
                loss = loss_fn(o, y[:, t:t + ch], mask)
                opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
                run.append(loss.item())
        r = evaluate(model, Xva, Yva, Mva)
        print(f'эпоха {ep + 1}/{epochs} | {time.time() - t0:.0f} с | loss {np.mean(run):.4f} | '
              f'valid все строки {r["все строки"][0]:.3f} {r["все строки"][1]} | is_scored {r["is_scored"][0]:.3f} {r["is_scored"][1]}', flush=True)
        if r['is_scored'][0] > best:
            best = r['is_scored'][0]; torch.save(model.state_dict(), f'{CKPT_DIR}/{tag}_best.pt')
    return model

# ---------- анализ по семействам и валидационной смеси ----------
@torch.no_grad()
def predict(model, X, idx, ch=2000, bs=49):
    model.eval(); out = []
    for s in range(0, len(idx), bs):
        x = torch.from_numpy(np.ascontiguousarray(X[idx[s:s + bs]])).to(dev); h = None; o_ = []
        for t in range(0, L, ch):
            o, h = model(x[:, t:t + ch], h); o_.append(o)
        out.append(torch.cat(o_, 1).clamp(-2, 2).cpu().numpy())
    return np.concatenate(out)

def sums_np(P, Y, M):
    post_m = np.broadcast_to(post, M.shape); out = {}
    for name, m in (('все строки', post_m), ('is_scored', M & post_m)):
        T = np.zeros((2, 6))
        for j in (0, 1):
            y = np.clip(Y[..., j][m], -2, 2).astype(np.float64); p = np.clip(P[..., j][m], -2, 2).astype(np.float64); a = np.abs(y)
            T[j] = [a.sum(), (a*y).sum(), (a*p).sum(), (a*y*y).sum(), (a*p*p).sum(), (a*y*p).sum()]
        out[name] = T
    return out

def corr(T):
    sw = T[:, 0]; my, mp = T[:, 1] / sw, T[:, 2] / sw
    cov = T[:, 5] / sw - my * mp
    den = (T[:, 3] / sw - my ** 2) * (T[:, 4] / sw - mp ** 2)
    return float(np.mean(np.where(den > 1e-16, cov / np.sqrt(np.maximum(den, 1e-16)), 0.0)))

def ens_mix(models):                        # скор по валидационной смеси семейств; нужны Xva, Yva, Mva, fam_v, have
    m = VALMIX / have; Sm = {}
    for f in range(4):
        ix = np.where(fam_v == f)[0]
        P = np.mean([predict(mod, Xva, ix) for mod in models], axis=0)
        Sm[f] = sums_np(P, Yva[ix], Mva[ix])
    return {k: round(corr(sum(m[f] * Sm[f][k] for f in range(4))), 3) for k in ('все строки', 'is_scored')}
mix_score = lambda model: ens_mix([model])

def code3(y):                               # код k в единицах семейства 4; насыщение = ±99
    k = np.round(y / U3).astype(np.int32); k[y >= 2 - 1e-6] = SAT; k[y <= -2 + 1e-6] = -SAT; return k

def wc(y, p, m):                            # взвешенная корреляция по маске m
    y, p = y[m].astype(np.float64), p[m].astype(np.float64); w = np.abs(y); sw = w.sum()
    my, mp = (w * y).sum() / sw, (w * p).sum() / sw
    cov, vy, vp = (w * (y - my) * (p - mp)).sum(), (w * (y - my) ** 2).sum(), (w * (p - mp) ** 2).sum()
    return cov / np.sqrt(vy * vp) if vp > 1e-12 else 0.0

Overwriting /content/drive/MyDrive/wnn/lib.py


In [15]:
%run -i /content/drive/MyDrive/wnn/lib.py

need = {'train': (600, 0, False), 'train3k': (3000, 0, False), 'valid300': (300, 1, True)}
def done(n): return all(os.path.exists(f'/content/data/{n}_{s}.npy') for s in (['X', 'Y'] + (['M'] if n.startswith('valid') else [])))
missing = [n for n in need if not done(n)]
print('нет файлов для:', missing or 'всё на месте')
if missing:
    if not os.path.exists(f'{ROOT}/train.parquet'):
        print('качаю данные (около 11 минут)...')
        subprocess.run("curl -sL https://files.wundernn.io/wnn_connectome_starterpack.tar.gz | "
                       "tar -xz --wildcards '*/baseline/*' '*/utils.py' '*/datasets/*.parquet' 2>/dev/null",
                       shell=True, cwd='/content')
    for n in missing:
        src = 'valid' if n.startswith('valid') else 'train'
        build(src, n, *need[n])
print(subprocess.run('df -h /content | tail -1', shell=True, capture_output=True, text=True).stdout)

нет файлов для: ['train', 'train3k', 'valid300']
train 0 0 с
train 500 62 с
train3k 0 0 с
train3k 500 60 с
train3k 1000 124 с
train3k 1500 190 с
train3k 2000 250 с
train3k 2500 314 с
valid300 0 0 с
overlay         113G   93G   21G  83% /



In [16]:
!pip install -q onnxruntime

In [ ]:
%run -i /content/drive/MyDrive/wnn/lib.py
import sys, time, importlib
from types import SimpleNamespace
ld = lambda n: np.load(f'/content/data/{n}.npy', mmap_mode='r')
Xva = ld('valid300_X'); Yva = np.load('/content/data/valid300_Y.npy'); Mva = np.load('/content/data/valid300_M.npy')
assert Xva.shape[-1] == 112, 'нужна 112-колоночная версия valid300'
fam_v = fam_of(Yva, verbose=False); have = np.bincount(fam_v, minlength=4) / len(fam_v)

# 1) baseline организаторов на valid300 (нужен onnxruntime: !pip install -q onnxruntime)
sys.path.insert(0, '/content/wnn_connectome_starterpack/baseline'); sol = importlib.import_module('solution')
mdl = sol.PredictionModel(); Pb = np.zeros((len(Xva), L, 2), np.float32); t = time.time()
for s in range(len(Xva)):
    Xs = np.ascontiguousarray(Xva[s])
    for st in range(L):
        r = mdl.predict(SimpleNamespace(seq_ix=s, step_in_seq=st, need_prediction=st >= WARM, state=Xs[st]))
        if r is not None: Pb[s, st] = r
    if s % 50 == 0: print(s, f'{time.time() - t:.0f} с', flush=True)
np.save(f'{CKPT_DIR}/Pb_valid300.npy', Pb)

def mix_of(P):
    m = VALMIX / have; Sm = {f: sums_np(P[np.where(fam_v == f)[0]], Yva[fam_v == f], Mva[fam_v == f]) for f in range(4)}
    return {k: round(corr(sum(m[f] * Sm[f][k] for f in range(4))), 3) for k in ('все строки', 'is_scored')}

body = Pb[:, WARM:].reshape(-1, 2)
print('baseline: std выходов', body.std(0).round(2), '| доля |p| >= 2:', (np.abs(body) >= 2).mean(0).round(3))
print('\nмасштаб (сдвиг 0):')
for s in (1.0, 0.7, 0.5, 0.4, 0.3, 0.25, 0.2, 0.15, 0.1): print(f'  x{s}:', mix_of(Pb * s))

print('\nсдвиг при масштабе 0.3 (одинаковый для обоих выходов):')
for b in (-0.3, -0.15, 0.0, 0.15, 0.3): print(f'  b={b:+.2f}:', mix_of(Pb * 0.3 + b))

# 2) среднее с нашей GRU (какой из чекпойнтов у тебя есть)
for ck in ('corr3k_1_best.pt', 'gru_full_val_best.pt', 'gru_w3k_best.pt'):
    if os.path.exists(f'{CKPT_DIR}/{ck}'):
        net = Net(128).to(dev); net.load_state_dict(torch.load(f'{CKPT_DIR}/{ck}', map_location=dev)); net.eval(); break
Pg = predict(net, Xva, np.arange(len(Xva)))
print('\nнаша GRU:', ck, mix_of(Pg))
for w in (0.0, 0.25, 0.5, 0.75, 1.0): print(f'  вес baseline x0.3 = {w}:', mix_of(w * Pb * 0.3 + (1 - w) * Pg))

0 2 с
50 99 с
100 197 с
150 295 с
200 393 с
250 490 с
baseline: std выходов [1.78 1.73] | доля |p| >= 2: [0.442 0.235]

масштаб (сдвиг 0):
  x1.0: {'все строки': 0.436, 'is_scored': 0.599}
  x0.7: {'все строки': 0.463, 'is_scored': 0.626}
  x0.5: {'все строки': 0.484, 'is_scored': 0.647}
  x0.4: {'все строки': 0.493, 'is_scored': 0.654}
  x0.3: {'все строки': 0.497, 'is_scored': 0.657}
  x0.25: {'все строки': 0.497, 'is_scored': 0.657}
  x0.2: {'все строки': 0.497, 'is_scored': 0.656}
  x0.15: {'все строки': 0.497, 'is_scored': 0.656}
  x0.1: {'все строки': 0.497, 'is_scored': 0.656}

сдвиг при масштабе 0.3 (одинаковый для обоих выходов):
  b=-0.30: {'все строки': 0.497, 'is_scored': 0.657}
  b=-0.15: {'все строки': 0.497, 'is_scored': 0.657}
  b=+0.00: {'все строки': 0.497, 'is_scored': 0.657}
  b=+0.15: {'все строки': 0.496, 'is_scored': 0.657}
  b=+0.30: {'все строки': 0.495, 'is_scored': 0.656}

наша GRU: corr3k_1_best.pt {'все строки': 0.49, 'is_scored': 0.654}
  вес baseline x0.3

In [ ]:
from scipy.signal import lfilter

def ema(P, span, start=WARM):
    if span <= 1: return P
    a = 2.0 / (span + 1); Q = P.copy(); x = P[:, start:, :].astype(np.float64)
    Q[:, start:, :] = lfilter([a], [1, -(1 - a)], x, axis=1, zi=(1 - a) * x[:, :1, :])[0]
    return Q

Pb3 = Pb * 0.3
variants = {'baseline x0.3': Pb3, 'наша GRU': Pg, 'среднее 50/50': 0.5 * Pb3 + 0.5 * Pg}
print('span: ' + ' | '.join(variants))
for span in (1, 3, 5, 10, 20, 40, 80, 160):
    row = [mix_of(ema(P, span)) for P in variants.values()]
    print(f'{span:4d}: ' + ' | '.join(f'{r["все строки"]:.3f} / {r["is_scored"]:.3f}' for r in row))

span: baseline x0.3 | наша GRU | среднее 50/50
   1: 0.497 / 0.657 | 0.490 / 0.654 | 0.501 / 0.660
   3: 0.493 / 0.653 | 0.486 / 0.650 | 0.497 / 0.656
   5: 0.488 / 0.648 | 0.481 / 0.646 | 0.492 / 0.651
  10: 0.479 / 0.636 | 0.472 / 0.635 | 0.482 / 0.639
  20: 0.464 / 0.615 | 0.458 / 0.616 | 0.467 / 0.620
  40: 0.445 / 0.584 | 0.439 / 0.587 | 0.448 / 0.590
  80: 0.422 / 0.542 | 0.418 / 0.548 | 0.425 / 0.549
 160: 0.398 / 0.493 | 0.396 / 0.502 | 0.402 / 0.502


In [ ]:
from sklearn.isotonic import IsotonicRegression
rng = np.random.default_rng(0); fold = np.zeros(len(Yva), int)
for f in range(4):
    ix = np.where(fam_v == f)[0]; rng.shuffle(ix); fold[ix[:len(ix) // 2]] = 1

def fit_iso(P, ix, step=7):
    maps = []
    for j in (0, 1):
        p = P[ix][:, WARM::step, j].ravel(); y = np.clip(Yva[ix][:, WARM::step, j], -2, 2).ravel(); w = np.abs(y); k = w > 0
        maps.append(IsotonicRegression(out_of_bounds='clip').fit(p[k], y[k], sample_weight=w[k]))
    return maps

def cross_iso(P):
    Q = P.copy()
    for k in (0, 1):
        tr, te = np.where(fold != k)[0], np.where(fold == k)[0]; maps = fit_iso(P, tr)
        for j in (0, 1): Q[te, :, j] = np.clip(maps[j].predict(P[te][..., j].ravel()).reshape(P[te][..., j].shape), -2, 2)
    return Q

Pb3 = Pb * 0.3
for name, P in {'baseline x0.3': Pb3, 'наша GRU': Pg, 'среднее 50/50': 0.5 * Pb3 + 0.5 * Pg}.items():
    print(f'{name:14s} до: {mix_of(P)} | после изотонической калибровки: {mix_of(cross_iso(P))}')

baseline x0.3  до: {'все строки': 0.497, 'is_scored': 0.657} | после изотонической калибровки: {'все строки': 0.495, 'is_scored': 0.654}
наша GRU       до: {'все строки': 0.49, 'is_scored': 0.654} | после изотонической калибровки: {'все строки': 0.488, 'is_scored': 0.652}
среднее 50/50  до: {'все строки': 0.501, 'is_scored': 0.66} | после изотонической калибровки: {'все строки': 0.499, 'is_scored': 0.657}


In [ ]:
Pb3 = Pb * 0.3
post_m = np.broadcast_to(post, Mva.shape); sel = Mva & post_m
Yc = np.clip(Yva, -2, 2).astype(np.float64); steps = np.arange(L)[None, :]
def sc(P, m): return np.mean([wc(Yc[..., j], np.clip(P[..., j], -2, 2).astype(np.float64), m) for j in (0, 1)])

print('--- по положению в последовательности (is_scored) ---')
for lo, hi in [(99, 1000), (1000, 3000), (3000, 8000), (8000, 20000)]:
    m = sel & (steps >= lo) & (steps < hi)
    print(f'шаги {lo:5d}-{hi:5d}: доля строк {m.sum() / sel.sum():.2f} | baseline×0.3 {sc(Pb3, m):.3f} | GRU {sc(Pg, m):.3f} | среднее {sc(0.5 * Pb3 + 0.5 * Pg, m):.3f}')
print('--- по семействам ---')
for f in range(4):
    m = sel & (fam_v[:, None] == f)
    print(f'семейство {f}: baseline×0.3 {sc(Pb3, m):.3f} | GRU {sc(Pg, m):.3f} | среднее {sc(0.5 * Pb3 + 0.5 * Pg, m):.3f}')

# стекинг: взвешенная МНК (веса |y|), подбор на одной половине последовательностей, проверка на другой
rng = np.random.default_rng(0); fold = np.zeros(len(Yva), int)
for f in range(4):
    ix = np.where(fam_v == f)[0]; rng.shuffle(ix); fold[ix[:len(ix) // 2]] = 1
def stack_eval(F):
    Q = np.zeros(Pg.shape, np.float32)
    for k in (0, 1):
        tr, te = np.where(fold != k)[0], np.where(fold == k)[0]; m = sel[tr]; Xf = F[tr][m].astype(np.float64)
        for j in (0, 1):
            y = Yc[tr][..., j][m]; w = np.abs(y)
            coef = np.linalg.solve(Xf.T @ (Xf * w[:, None]) + 1e-6 * np.eye(Xf.shape[1]), Xf.T @ (y * w))
            Q[te, :, j] = np.clip(F[te] @ coef, -2, 2)
    return Q
one = np.ones(Pb.shape[:2] + (1,), np.float32)
print('\n--- стекинг ---')
print('baseline×0.3 как есть:      ', mix_of(Pb3))
print('смесь двух выходов baseline:', mix_of(stack_eval(np.concatenate([Pb3, one], -1))))
print('baseline + GRU, 4 выхода:   ', mix_of(stack_eval(np.concatenate([Pb3, Pg, one], -1))))

--- по положению в последовательности (is_scored) ---
шаги    99- 1000: доля строк 0.04 | baseline×0.3 0.731 | GRU 0.715 | среднее 0.729
шаги  1000- 3000: доля строк 0.09 | baseline×0.3 0.636 | GRU 0.639 | среднее 0.642
шаги  3000- 8000: доля строк 0.24 | baseline×0.3 0.682 | GRU 0.682 | среднее 0.687
шаги  8000-20000: доля строк 0.62 | baseline×0.3 0.669 | GRU 0.664 | среднее 0.671
--- по семействам ---
семейство 0: baseline×0.3 0.780 | GRU 0.773 | среднее 0.782
семейство 1: baseline×0.3 0.755 | GRU 0.748 | среднее 0.756
семейство 2: baseline×0.3 0.719 | GRU 0.701 | среднее 0.715
семейство 3: baseline×0.3 0.399 | GRU 0.398 | среднее 0.403

--- стекинг ---
baseline×0.3 как есть:       {'все строки': 0.497, 'is_scored': 0.657}
смесь двух выходов baseline: {'все строки': 0.491, 'is_scored': 0.655}
baseline + GRU, 4 выхода:    {'все строки': 0.495, 'is_scored': 0.658}


In [ ]:
from scipy.signal import lfilter
NF0 = 112; COLS = list(range(0, 52)) + list(range(96, 104)); SPANS = (10, 50, 200, 1000); K = len(COLS) * len(SPANS)
!df -h /content | tail -1
def ms_feats(X):
    x = np.asarray(X[:, COLS], np.float64); out = []
    for s in SPANS:
        a = 2.0 / (s + 1)
        out.append(x - lfilter([a], [1, -(1 - a)], x, axis=0, zi=(1 - a) * x[:1])[0])
    return np.concatenate(out, 1).astype(np.float32)

ld = lambda n: np.load(f'/content/data/{n}.npy', mmap_mode='r')
src_tr, src_va = ld('train_X'), ld('valid300_X'); assert src_tr.shape[-1] == 112 and src_va.shape[-1] == 112
sd = np.concatenate([ms_feats(src_tr[i]) for i in range(0, 600, 20)]).std(0) + 1e-3
np.save(f'{CKPT_DIR}/ms_sd.npy', sd)
def build_ms(src, name):
    out = np.lib.format.open_memmap(f'/content/data/{name}.npy', 'w+', np.float16, (len(src), L, NF0 + K))
    for i in range(len(src)):
        x = np.asarray(src[i]); out[i, :, :NF0] = x; out[i, :, NF0:] = ms_feats(x) / sd
    out.flush()
build_ms(src_tr, 'train_ms16'); build_ms(src_va, 'valid300_ms16')

class V16:
    def __init__(s, X): s.X = X
    def __len__(s): return len(s.X)
    def __getitem__(s, i): return np.asarray(s.X[i]).astype(np.float32)
Xtr = V16(ld('train_ms16')); Ytr = np.load('/content/data/train_Y.npy')
Xva = V16(ld('valid300_ms16')); Yva = np.load('/content/data/valid300_Y.npy'); Mva = np.load('/content/data/valid300_M.npy')
fam_v = fam_of(Yva, verbose=False); have = np.bincount(fam_v, minlength=4) / len(fam_v)
loss_fn = lambda p, y, mask: (y.clamp(-2, 2).abs() * (p - y.clamp(-2, 2)) ** 2)[mask].mean()
for seed in (0, 1):
    torch.manual_seed(seed); m = train(Net2(NF0 + K).to(dev), epochs=8, tag=f'ms_{seed}')
    print('>>> признаки истории, сид', seed, mix_score(m), '| контроль 0.467/0.640 и 0.469/0.641', flush=True)

overlay         113G   66G   47G  59% /
эпоха 1/8 | 98 с | loss 1.0950 | valid все строки 0.445 [0.45 0.44] | is_scored 0.645 [0.637 0.654]
эпоха 2/8 | 95 с | loss 1.0333 | valid все строки 0.459 [0.461 0.458] | is_scored 0.639 [0.634 0.644]
эпоха 3/8 | 94 с | loss 1.0047 | valid все строки 0.457 [0.462 0.453] | is_scored 0.649 [0.644 0.654]
эпоха 4/8 | 94 с | loss 0.9883 | valid все строки 0.443 [0.441 0.445] | is_scored 0.642 [0.638 0.646]
эпоха 5/8 | 94 с | loss 0.9540 | valid все строки 0.451 [0.457 0.445] | is_scored 0.646 [0.642 0.65 ]
эпоха 6/8 | 94 с | loss 0.9089 | valid все строки 0.437 [0.448 0.426] | is_scored 0.638 [0.634 0.642]
эпоха 7/8 | 94 с | loss 0.8375 | valid все строки 0.420 [0.429 0.412] | is_scored 0.625 [0.621 0.629]
эпоха 8/8 | 93 с | loss 0.7989 | valid все строки 0.405 [0.41  0.401] | is_scored 0.614 [0.608 0.619]
>>> признаки истории, сид 0 {'все строки': 0.406, 'is_scored': 0.6} | контроль 0.467/0.640 и 0.469/0.641
эпоха 1/8 | 86 с | loss 1.0983 | valid вс

Следующая ячейка нужна, чтобы ноутбук заново не гонять

In [17]:
%run -i /content/drive/MyDrive/wnn/lib.py
import os

ld = lambda n: np.load(f'/content/data/{n}.npy', mmap_mode='r')

# --- валидация и метки ---
Xva = ld('valid300_X')
Yva = np.load('/content/data/valid300_Y.npy')
Mva = np.load('/content/data/valid300_M.npy')
assert Xva.shape[-1] == 112, 'нужна 112-колоночная версия valid300'
fam_v = fam_of(Yva, verbose=False)
have = np.bincount(fam_v, minlength=4) / len(fam_v)

# --- baseline из сохранённого файла ---
Pb = np.load(f'{CKPT_DIR}/Pb_valid300.npy')
assert Pb.shape == (len(Xva), L, 2), Pb.shape
Pb3 = Pb * 0.3

# --- наша GRU из чекпойнта ---
net = Net(128).to(dev)
net.load_state_dict(torch.load(f'{CKPT_DIR}/corr3k_1_best.pt', map_location=dev))
net.eval()
Pg = predict(net, Xva, np.arange(len(Xva)))

# --- вспомогательные переменные для последней ячейки ---
post_m = np.broadcast_to(post, Mva.shape)
sel = Mva & post_m
Yc = np.clip(Yva, -2, 2).astype(np.float64)

def mix_of(P):
    m = VALMIX / have
    Sm = {f: sums_np(P[np.where(fam_v == f)[0]], Yva[fam_v == f], Mva[fam_v == f]) for f in range(4)}
    return {k: round(corr(sum(m[f] * Sm[f][k] for f in range(4))), 3) for k in ('все строки', 'is_scored')}

print('baseline×0.3:', mix_of(Pb3))
print('GRU         :', mix_of(Pg))
print('среднее     :', mix_of(0.5 * Pb3 + 0.5 * Pg))

baseline×0.3: {'все строки': 0.497, 'is_scored': 0.657}
GRU         : {'все строки': 0.49, 'is_scored': 0.654}
среднее     : {'все строки': 0.501, 'is_scored': 0.66}


In [ ]:
Pens = 0.5 * Pb3 + 0.5 * Pg; ix3 = np.where(fam_v == 3)[0]
s3 = sel[ix3]; c = Pens[ix3][s3].std() / Yc[ix3][s3].std()
print(f'F3: масштаб предсказаний к таргету {c:.2f}')
print('λ | корреляция внутри F3 (is_scored) | общий скор по валидационной смеси')
for lam in (0.0, 0.1, 0.2, 0.3, 0.45, 0.6):
    Q = Pens.copy(); Q[ix3] = (1 - lam) * Pens[ix3] + lam * c * Yc[ix3]
    f3 = np.mean([wc(Yc[ix3][..., j], np.clip(Q[ix3][..., j], -2, 2).astype(np.float64), s3) for j in (0, 1)])
    print(f'{lam:.2f} | {f3:.3f} | {mix_of(Q)}')

F3: масштаб предсказаний к таргету 0.56
λ | корреляция внутри F3 (is_scored) | общий скор по валидационной смеси
0.00 | 0.403 | {'все строки': 0.501, 'is_scored': 0.66}
0.10 | 0.531 | {'все строки': 0.533, 'is_scored': 0.692}
0.20 | 0.651 | {'все строки': 0.563, 'is_scored': 0.722}
0.30 | 0.756 | {'все строки': 0.59, 'is_scored': 0.75}
0.45 | 0.875 | {'все строки': 0.626, 'is_scored': 0.786}
0.60 | 0.947 | {'все строки': 0.655, 'is_scored': 0.817}


In [ ]:
offs = np.array([-200, -100, -50, -20, -10, 0, 10, 20, 50, 100, 200])
for j in (0, 1):
    e = code3(Yc[ix3, :, j]) <= -3                              # событие: -3 или насыщенный минус
    on = np.pad(e[:, 1:] & ~e[:, :-1], ((0, 0), (1, 0)))
    si, ti = np.where(on); ok = (ti >= WARM + 200) & (ti < L - 200); si, ti = si[ok], ti[ok]
    Wy = np.stack([Yc[ix3][si, ti + o, j] for o in offs], 1); Wp = np.stack([Pens[ix3][si, ti + o, j] for o in offs], 1)
    print(f't{j}: {len(si)} начал событий | среднее предсказание по всем строкам F3: {Pens[ix3][..., j].mean():.2f}')
    print('смещение:', offs); print('таргет :', np.round(Wy.mean(0), 2)); print('прогноз:', np.round(Wp.mean(0), 2))

t0: 2305 начал событий | среднее предсказание по всем строкам F3: -0.25
смещение: [-200 -100  -50  -20  -10    0   10   20   50  100  200]
таргет : [-0.47 -0.48 -0.54 -0.64 -0.68 -1.97 -1.67 -1.57 -1.32 -0.97 -0.69]
прогноз: [-0.28 -0.29 -0.3  -0.27 -0.21 -0.45 -0.36 -0.41 -0.43 -0.4  -0.32]
t1: 2283 начал событий | среднее предсказание по всем строкам F3: -0.36
смещение: [-200 -100  -50  -20  -10    0   10   20   50  100  200]
таргет : [-0.5  -0.51 -0.57 -0.65 -0.66 -1.97 -1.63 -1.55 -1.3  -0.99 -0.73]
прогноз: [-0.42 -0.42 -0.41 -0.4  -0.34 -0.57 -0.46 -0.5  -0.55 -0.51 -0.47]


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
names = ([f'i0_p{k}' for k in range(22)] + [f'i0_v{k}' for k in range(22)] + [f'i0_dp{k}' for k in range(4)] + [f'i0_dv{k}' for k in range(4)]
         + [f'i1_p{k}' for k in range(22)] + [f'i1_v{k}' for k in range(22)] + [f'i1_dp{k}' for k in range(4)] + [f'i1_dv{k}' for k in range(4)]
         + [f'a{k}' for k in range(8)]); assert len(names) == 112

# --- часть 1: меняются ли входы в момент начала события (F3, t0) ---
Xraw = np.load('/content/data/valid300_X.npy', mmap_mode='r'); assert Xraw.shape[-1] == 112
X3 = np.asarray(Xraw[ix3]); dX = np.abs(np.diff(X3, axis=1, prepend=X3[:, :1])); bg = dX[:, WARM:].mean((0, 1)) + 1e-9
e = Yc[ix3, :, 0] <= -1.5; on = np.pad(e[:, 1:] & ~e[:, :-1], ((0, 0), (1, 0)))
si, ti = np.where(on); ok = (ti >= WARM + 10) & (ti < L - 10); si, ti = si[ok], ti[ok]
print(len(si), 'начал событий t0 в F3 (валидация)')
for o in (-5, 0, 1, 5):
    r = dX[si, ti + o].mean(0) / bg; top = np.argsort(-r)[:8]
    print(f'смещение {o:+d}: ' + ', '.join(f'{names[j]} x{r[j]:.1f}' for j in top) + f' | медиана по колонкам x{np.median(r):.2f}')
del X3, dX

# --- часть 2: бустинг с разностями на лагах для F3 против наших моделей ---
LAGS = (1, 5, 20, 100); LC = list(range(0, 52)) + list(range(96, 104))
def lag_feats(x):
    out = [x] + [x[:, LC] - np.concatenate([np.repeat(x[:1, LC], lag, 0), x[:-lag, LC]], 0) for lag in LAGS]
    return np.concatenate(out, 1).astype(np.float32)

Xt = np.load('/content/data/train_X.npy', mmap_mode='r'); Yt = np.load('/content/data/train_Y.npy'); assert Xt.shape[-1] == 112
it3 = np.where(fam_of(Yt, verbose=False) == 3)[0]; rng = np.random.default_rng(0); A, B = [], []
for i in it3:
    f = lag_feats(np.asarray(Xt[i])); r = rng.choice(np.arange(WARM, L), 1500, replace=False); A.append(f[r]); B.append(np.clip(Yt[i][r], -2, 2))
A, B = np.concatenate(A), np.concatenate(B); print('обучающих строк F3:', A.shape)
gb = [HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, early_stopping=False, random_state=0).fit(A, B[:, j], sample_weight=np.abs(B[:, j])) for j in (0, 1)]
Pgb = np.zeros(Pens[ix3].shape, np.float32)
for k, i in enumerate(ix3):
    f = lag_feats(np.asarray(Xraw[i]))
    for j in (0, 1): Pgb[k, :, j] = gb[j].predict(f)

s3 = sel[ix3]; Y3c = Yc[ix3]
f3sc = lambda P: [round(wc(Y3c[..., j], np.clip(P[..., j], -2, 2).astype(np.float64), s3), 3) for j in (0, 1)]
for n, P in {'baseline×0.3': Pb3[ix3], 'наша GRU': Pg[ix3], 'среднее двух': Pens[ix3], 'бустинг + история': Pgb}.items(): print(f'{n:18s} F3 (t0, t1): {f3sc(P)}')
z = lambda P: P / P[s3].std()
print('среднее двух + бустинг:', f3sc((z(Pens[ix3]) + z(Pgb)) / 2))

2346 начал событий t0 в F3 (валидация)
смещение -5: i0_dv1 x1.9, i0_dv0 x1.7, i0_dv3 x1.7, i0_p5 x1.7, i0_v5 x1.5, i0_v8 x1.5, i0_v2 x1.5, a5 x1.5 | медиана по колонкам x0.93
смещение +0: i0_v2 x8.2, i0_v4 x7.6, i0_v9 x7.2, i0_v5 x6.8, i0_v6 x6.8, i0_v0 x6.4, i0_v3 x6.0, i0_v1 x5.8 | медиана по колонкам x0.89
смещение +1: i0_p4 x4.9, i0_dv1 x4.9, i0_v10 x4.7, i0_p12 x4.3, i0_p19 x3.9, i0_v7 x3.8, i0_p17 x3.7, a6 x3.7 | медиана по колонкам x0.38
смещение +5: i0_v2 x1.5, i0_p5 x1.4, a3 x1.3, i0_v8 x1.3, i0_v5 x1.3, i0_v1 x1.3, i0_v4 x1.3, i0_v3 x1.3 | медиана по колонкам x0.67
обучающих строк F3: (412500, 352)
baseline×0.3       F3 (t0, t1): [np.float64(0.41), np.float64(0.388)]
наша GRU           F3 (t0, t1): [np.float64(0.406), np.float64(0.389)]
среднее двух       F3 (t0, t1): [np.float64(0.413), np.float64(0.394)]
бустинг + история  F3 (t0, t1): [np.float64(0.361), np.float64(0.393)]
среднее двух + бустинг: [np.float64(0.404), np.float64(0.411)]


In [22]:
# ============================================================
# ПОСЛЕДНЯЯ ЯЧЕЙКА: детектор начала события F3
# Сама готовит все переменные (ix3, Yc, sel, Xraw), не зависит от прошлых ячеек
# ============================================================
%run -i /content/drive/MyDrive/wnn/lib.py
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve

ld = lambda n: np.load(f'/content/data/{n}.npy', mmap_mode='r')

# --- готовим данные ---
Xraw = ld('valid300_X')
Yva = np.load('/content/data/valid300_Y.npy')
Mva = np.load('/content/data/valid300_M.npy')
assert Xraw.shape[-1] == 112, 'нужна 112-колоночная версия valid300'

fam_v = fam_of(Yva, verbose=False)
have = np.bincount(fam_v, minlength=4) / len(fam_v)
ix3 = np.where(fam_v == 3)[0]                     # <-- ВОТ ОНО
Yc = np.clip(Yva, -2, 2).astype(np.float64)
post_m = np.broadcast_to(post, Mva.shape)
sel = Mva & post_m

# --- сама диагностика ---
X3 = np.asarray(Xraw[ix3]).astype(np.float32)     # (98, L, 112)
sdX = np.diff(X3, axis=1, prepend=X3[:, :1]); dX = np.abs(sdX)
e0, e1 = code3(Yc[ix3, :, 0]) <= -3, code3(Yc[ix3, :, 1]) <= -3
starts = lambda e: np.pad(e[:, 1:] & ~e[:, :-1], ((0, 0), (1, 0)))
on0, on1 = starts(e0), starts(e1); on = on0 | on1
vt = np.zeros(L, bool); vt[WARM + 1:] = True
print('начал событий:', on[:, vt].sum(), '| доля шагов:', round(on[:, vt].mean(), 5))

rows = np.arange(len(ix3)); tr_s, te_s = rows[rows % 2 == 0], rows[rows % 2 == 1]
F = lambda s: np.concatenate([dX[s][:, vt], X3[s][:, vt]], -1).reshape(-1, 224)
Ftr, ytr = F(tr_s), on[tr_s][:, vt].reshape(-1)
rng = np.random.default_rng(0); idx = np.concatenate([np.where(ytr)[0], rng.choice(np.where(~ytr)[0], 200000, replace=False)])
clf = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, random_state=0).fit(Ftr[idx], ytr[idx])
yte = on[te_s][:, vt].reshape(-1); p = clf.predict_proba(F(te_s))[:, 1]
pr, rc, _ = precision_recall_curve(yte, p)
print(f'\nобнаружение начала события: AUC {roc_auc_score(yte, p):.3f} | доля событий среди всех шагов {yte.mean():.5f}')
for r in (0.3, 0.5, 0.7, 0.9): print(f'  полнота {r}: точность {pr[np.where(rc >= r)[0].max()]:.3f}')

pos = np.arange(L)[None, :]; si, ti = np.where((on0 ^ on1) & (pos >= WARM + 1) & (pos < L - 3))
yside = on0[si, ti].astype(int); S = np.concatenate([sdX[si, ti + o] for o in (0, 1, 2)], 1); acc = []
for a, b in ((0, 1), (1, 0)):
    mtr, mte = si % 2 == a, si % 2 == b
    c2 = HistGradientBoostingClassifier(max_iter=100, max_depth=3, learning_rate=0.1, random_state=0).fit(S[mtr], yside[mtr])
    acc.append((c2.predict(S[mte]) == yside[mte]).mean())
print(f'\nсторона события (t0 или t1 ныряет) по знаковым разностям на t..t+2: точность {np.mean(acc):.3f} | доля класса t0 {yside.mean():.2f} | событий {len(si)}')

начал событий: 4691 | доля шагов: 0.00241

обнаружение начала события: AUC 0.985 | доля событий среди всех шагов 0.00224
  полнота 0.3: точность 0.124
  полнота 0.5: точность 0.112
  полнота 0.7: точность 0.076
  полнота 0.9: точность 0.050

сторона события (t0 или t1 ныряет) по знаковым разностям на t..t+2: точность 0.999 | доля класса t0 0.50 | событий 4688
